# Turbomachinery Degradation & RUL Prognostics with Deep LSTM Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedbasra-star/turbomachinery-degradation-lstm/blob/main/notebooks/rul_quickstart_demo.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-brightgreen.svg)](https://www.python.org/downloads/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0+-ee4c2c.svg)](https://pytorch.org/)

## 1. Turbomachinery Prognostic Background & Physics
In gas turbines and jet aircraft turbofans, rotating components (e.g. high-pressure compressor blades, bearings, combustor liners) experience continuous mechanical, aerodynamic, and thermal degradation. 

The **NASA C-MAPSS (Commercial Modular Aero-Propulsion System Simulation)** benchmark dataset models the degradation of 100 run-to-failure engines. 

### Asymmetric Industrial Safety Metric (NASA Scoring Function)
In aviation and turbomachinery maintenance, late predictions ($y_{pred} > y_{true}$) risk catastrophic in-flight failure and are penalized much more heavily than early predictions ($y_{pred} < y_{true}$):
$$S = \sum_{i=1}^{N} s_i, \quad s_i = \begin{cases} e^{-\frac{d_i}{13}} - 1 & \text{if } d_i < 0 \text{ (Early Maintenance / Conservative)} \\ e^{\frac{d_i}{10}} - 1 & \text{if } d_i \ge 0 \text{ (Late Prediction / Critical Safety Hazard)} \end{cases}$$
where $d_i = y_{pred} - y_{true}$.

In [ ]:
# Install dependencies if running on Google Colab
import sys
try:
    import torch, pandas, numpy, matplotlib, seaborn
except ImportError:
    !pip install -q torch pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Device: {device}")

## 2. Ingesting & Visualizing Sensor Degradation Trajectories

In [ ]:
# Load C-MAPSS data from repo
import os
train_path = "../data/train_FD001.txt"
if os.path.exists(train_path):
    from data.dataset import load_cmapss_data, prepare_sliding_windows, INFORMATIVE_SENSORS
    train_df, test_df, true_rul = load_cmapss_data("../data")
else:
    # Fallback to local import
    from data.dataset import load_cmapss_data, prepare_sliding_windows, INFORMATIVE_SENSORS
    train_df, test_df, true_rul = load_cmapss_data()

print(f"Train Engines: {train_df['unit_nr'].nunique()} | Test Engines: {test_df['unit_nr'].nunique()}")
train_df.head()

In [ ]:
# Plot High Pressure Compressor (HPC) Outlet Temperature T30 (s_3) for first 4 engines
plt.figure(figsize=(9, 4.5))
for u in [1, 2, 5, 8]:
    sub = train_df[train_df["unit_nr"] == u]
    plt.plot(sub["time_cycles"], sub["s_3"], label=f"Engine #{u}", linewidth=1.8)

plt.xlabel("Operational Flight Cycles", fontweight="bold")
plt.ylabel("HPC Outlet Temperature T30 (°R)", fontweight="bold")
plt.title("Thermodynamic Degradation: HPC Temperature Drift Over Engine Lifespan", fontweight="bold")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

## 3. Sliding Window Preprocessing & Deep Model Inference

In [ ]:
data_dict = prepare_sliding_windows(train_df, test_df, true_rul, sequence_length=30, max_rul=125.0)
X_train, y_train = data_dict["X_train"], data_dict["y_train"]
X_test, y_test = data_dict["X_test"], data_dict["y_test"]
print(f"Sequenced Train Windows: {X_train.shape} | Test Fleet Windows: {X_test.shape}")

## 4. CNN-LSTM Hybrid Architecture Definition

In [ ]:
class QuickCNNLSTM(nn.Module):
    def __init__(self, in_features=14, hidden_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=in_features, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        # [batch, seq_len, feats] -> conv on features
        x_conv = self.conv(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.lstm(x_conv)
        return self.fc(out[:, -1, :])

model = QuickCNNLSTM().to(device)
print(model)

## 5. Single Engine Continuous Lifecycle Simulation

In [ ]:
# Load trained checkpoint if available
ckpt_path = "../checkpoints/best_model.pt"
if os.path.exists(ckpt_path):
    from models.architectures import build_model
    trained_net = build_model("CNN-LSTM Hybrid")
    trained_net.load_state_dict(torch.load(ckpt_path, map_location=device))
    trained_net.eval()
    
    with torch.no_grad():
        preds = trained_net(torch.tensor(X_test).to(device)).cpu().numpy().ravel()
        rmse = np.sqrt(np.mean((preds - y_test)**2))
        print(f"Evaluated Checkpoint -> Fleet Test RMSE: {rmse:.2f} Flight Cycles")
else:
    print("Train checkpoint will be evaluated when train.py completes.")